In [7]:
# Daily coverage table 
cov = clean.groupby("collection_date").agg(total_obs=("title", "size"), unique_listings=("listing_key", "nunique"),
                                           median_price=("price_num", "median"), missing_condition=("condition_clean", lambda s: int(s.isna().sum())),
                                           missing_posting_date=("posting_date", lambda s: int(s.isna().sum())))
for cat in CATS:
    cov[cat] = clean[clean.category == cat].groupby("collection_date").size()
cov[CATS] = cov[CATS].fillna(0).astype(int)      # 0 here = no records collected for that run
cov = cov[["total_obs", "Mobile Phones", "Furniture", "Fashion", "unique_listings", "median_price", "missing_condition", "missing_posting_date"]]
cov.index = cov.index.strftime("%Y-%m-%d")
display(cov)

# Evidence from the scraper log for the incomplete days
if LOG_PATH.exists():
    parts = [ln.rstrip("\n").split(" | ", 2) for ln in open(LOG_PATH, encoding="utf-8", errors="ignore") if " | " in ln]
    log = pd.DataFrame([p for p in parts if len(p) == 3], columns=["time", "level", "message"])
    err = log[log["level"].str.strip() == "ERROR"].copy()
    err["date"] = err["time"].str[:10]; err["category"] = err["message"].str.extract(r"Request error in ([A-Za-z ]+):")[0]
    err["cause"] = np.where(err["message"].str.contains("NameResolution|getaddrinfo"), "DNS lookup failed",
                    np.where(err["message"].str.contains("timed out"), "timeout", "other"))
    print("\nScraper-log ERROR lines by date, category and cause:")
    display(err.groupby(["date", "category", "cause"]).size().rename("errors").reset_index())

NameError: name 'clean' is not defined

In [ ]:
n_obs, n_ids = len(df), df["listing_key"].nunique()
print(f"Total observations (rows): {n_obs:,} | unique listings (listing_key): {n_ids:,} | unique advert_id/URL: {df['advert_id'].nunique():,}")
print(f"A listing seen on several days counts once as a listing but once per day as an observation.\n")

summary = pd.DataFrame({"observations": df.groupby("category").size(), "unique_listings": df.groupby("category")["listing_key"].nunique()}).loc[CATS]
display(summary)
price = df["price_num"]
print(f"Overall price (KSh): median {price.median():,.0f} | mean {price.mean():,.0f} | min {price.min():,.0f} | max {price.max():,.0f} | std {price.std():,.0f} | IQR {price.quantile(.75)-price.quantile(.25):,.0f}")
print("\nCondition (cleaned) share of observations, %:"); print((df["condition_clean"].value_counts(dropna=False, normalize=True) * 100).round(1).to_string())
print("\nMost common locations (area), observations:"); print(df["location"].value_counts().head(8).to_string())
print("\nCounty share, %:"); print((df["county"].value_counts(normalize=True).head(4) * 100).round(1).to_string())

# Chart EDA1: observations vs unique listings per category
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(CATS)); w = .38
b1 = ax.bar(x - w/2, summary["observations"], w, label="Observations (rows, all 7 days)", color=[PAL[c] for c in CATS], alpha=.55)
b2 = ax.bar(x + w/2, summary["unique_listings"], w, label="Unique listings (distinct listing_key)", color=[PAL[c] for c in CATS])
for b in list(b1) + list(b2): ax.annotate(f"{int(b.get_height()):,}", (b.get_x() + b.get_width()/2, b.get_height()), ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(CATS); ax.set_ylabel("Count"); ax.set_xlabel("Category")
ax.set_title("Collected records per category: observations vs unique listings")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [5]:
def category_stats(g):
    q1, q3 = g.quantile(.25), g.quantile(.75); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({"n": g.size, "mean": g.mean(), "median": g.median(), "std": g.std(), "min": g.min(), "Q1": q1, "Q3": q3, "max": g.max(),
                      "IQR": iqr, "lower_fence": lo, "upper_fence": hi, "low_outliers": int((g < lo).sum()), "high_outliers": int((g > hi).sum()),
                      "outlier_%": 100 * ((g < lo) | (g > hi)).mean(), "skewness": stats.skew(g), "CV_%": 100 * g.std() / g.mean()})
price_stats = df.groupby("category")["price_num"].apply(category_stats).unstack().loc[CATS]
display(price_stats.T)

# Which listings are the outliers?  Inspect before deciding anything.
fence = price_stats["upper_fence"]
out_high = df[df["price_num"] > df["category"].map(fence)]
print("High outliers: count and distinct listings by category")
print(pd.DataFrame({"observations": out_high.groupby("category").size(), "listings": out_high.groupby("category")["listing_key"].nunique()}).loc[CATS].to_string())
for cat in CATS:
    print(f"\n--- {cat}: five most expensive outlier observations ---")
    display(out_high[out_high.category == cat].nlargest(5, "price_num")[["title", "price_num", "condition_clean", "collection_date"]].assign(collection_date=lambda d: d.collection_date.dt.date))
low_out = df[df["price_num"] < df["category"].map(price_stats["lower_fence"])]
print("Low outliers (below Q1 - 1.5*IQR):", len(low_out))

# Sensitivity check: the same medians using one row per listing (its latest observation)
latest = df.sort_values(["collection_date", "scraped_dt"]).groupby("listing_key").tail(1)
print("\nMedian price using one row per unique listing (latest observation):")
print(latest.groupby("category")["price_num"].agg(["count", "median"]).loc[CATS].to_string())

NameError: name 'df' is not defined

In [4]:
# Chart A1: box plots, linear scale (left) and logarithmic scale (right)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
labels = [f"{c}\n(n={int(price_stats.loc[c, 'n']):,})" for c in CATS]
for ax, log in zip(axes, [False, True]):
    sns.boxplot(data=df, x="category", y="price_num", order=CATS, palette=PAL, hue="category", legend=False, ax=ax,
                flierprops=dict(marker="o", markersize=3, alpha=.4), linewidth=1.2)
    ax.set_xticks(range(3)); ax.set_xticklabels(labels)
    ax.yaxis.set_major_formatter(KSH); ax.set_xlabel("Category"); ax.set_ylabel("Price (KSh)" + (", log scale" if log else ""))
    if log: ax.set_yscale("log")
    for i, c in enumerate(CATS):
        ax.text(i, price_stats.loc[c, "median"], f" {price_stats.loc[c, 'median']:,.0f}", va="bottom", ha="left", fontsize=8, color="black", fontweight="bold")
axes[0].set_title("Price by category (linear scale)"); axes[1].set_title("Price by category (log scale)")
fig.suptitle("A1. Listing price distribution by category - box = IQR, line = median, dots = outliers (1.5 x IQR rule)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

In [6]:
# Chart A2: histograms with log-spaced bins (prices are strongly right-skewed); nothing is trimmed
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=False)
for ax, c in zip(axes, CATS):
    v = df.loc[df.category == c, "price_num"]
    bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 30)
    ax.hist(v, bins=bins, color=PAL[c], edgecolor="white")
    ax.set_xscale("log"); ax.xaxis.set_major_formatter(KSH)
    ax.axvline(v.median(), color="black", ls="--", lw=1.2, label=f"median {v.median():,.0f}")
    ax.axvline(v.mean(), color="red", ls=":", lw=1.5, label=f"mean {v.mean():,.0f}")
    ax.set_title(c); ax.set_xlabel("Price (KSh, log scale)"); ax.set_ylabel("Number of observations"); ax.legend(frameon=False)
fig.suptitle("A2. Distribution of listing prices within each category (log-spaced bins, all values shown)", fontsize=12, y=1.03)
plt.tight_layout(); plt.show()
print(price_stats[["mean", "median", "skewness"]].assign(mean_over_median=lambda d: d["mean"] / d["median"]).round(2).to_string())

NameError: name 'plt' is not defined